In [2]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 10.7 MB/s  0:00:00 eta 0:00:01


In [3]:
import pretty_midi
import music21
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
import json

# Configuration dictionary
CONFIG = {
    # Data directories
    "RAW_DIR": "data/raw",
    "PROCESSED_DIR": "data/processed",
    "OUTPUT_DIR": "outputs/remixes",

    # MIDI parameters
    "TICKS_PER_BEAT": 4,  # 16th note resolution
    "TARGET_KEY": "C",

    # Model training parameters
    "MAX_SEQ_LEN": 512,
    "BATCH_SIZE": 32,
    "EPOCHS": 100,
    "LR": 1e-3,

    # Model architecture
    "EMBED_DIM": 128,
    "NUM_HEADS": 4,
    "NUM_LAYERS": 3,
}


In [4]:
def find_midi_pairs(raw_dir):
    """
    Scan raw_dir for MIDI file pairs following naming convention:
    Kit_N_Role_120BPM_Key.mid

    Groups files by kit number and pairs "Chord" with "Lead" files.
    Returns list of dicts with keys: "chords", "melody", "key"
    """
    raw_path = Path(raw_dir)
    midi_files = list(raw_path.glob("*.mid"))

    # Group files by kit number
    kits = {}
    for file in midi_files:
        name = file.stem  # filename without .mid
        parts = name.split("_")

        # Extract kit number (second part: Kit_N)
        if len(parts) >= 2 and parts[0] == "Kit":
            kit_num = parts[1]
            if kit_num not in kits:
                kits[kit_num] = {"Chord": None, "Lead": None, "key": None}

            # Extract role (Chord or Lead)
            if len(parts) >= 3:
                role = parts[2]
                if role in ["Chord", "Lead"]:
                    kits[kit_num][role] = str(file)

            # Extract key (last part before .mid)
            if len(parts) >= 5:
                kits[kit_num]["key"] = parts[4]

    # Build pairs where both Chord and Lead exist
    pairs = []
    for kit_num, data in kits.items():
        if data["Chord"] and data["Lead"] and data["key"]:
            pairs.append({
                "chords": data["Chord"],
                "melody": data["Lead"],
                "key": data["key"]
            })

    print(f"Found {len(pairs)} MIDI pairs")
    return pairs

In [5]:
def transpose_to_c(midi_path, source_key):
    """
    Load a MIDI file, transpose it to C major using the provided source_key.

    Args:
        midi_path: Path to the MIDI file
        source_key: Source key as a string (e.g. "G", "E")

    Returns:
        Transposed music21 Score object, or None if parsing fails
    """
    try:
        # Load the MIDI file
        score = music21.converter.parse(midi_path)

        # Parse source key
        source_key_obj = music21.key.Key(source_key)
        target_key_obj = music21.key.Key("C")

        # Calculate interval needed to transpose to C major
        interval = music21.interval.Interval(source_key_obj.tonic, target_key_obj.tonic)

        # Transpose all notes by that interval
        transposed_score = score.transpose(interval)

        return transposed_score
    except Exception as e:
        print(f"Error transposing {midi_path}: {e}")
        return None

In [6]:
def extract_note_sequence(score, part_name):
    """
    Extract note sequence from a music21 Score.

    Args:
        score: music21 Score object
        part_name: "melody" or "chords" (for identification, not used in extraction)

    Returns:
        List of dicts with keys: pitch (int or list of ints), duration (float), offset (float)
        All quantized to nearest 16th note (0.25 quarter note units)
    """
    notes_list = []
    sixteenth = 0.25  # 16th note in quarter note units

    # Get first part from score
    if len(score.parts) == 0:
        part = score.flatten()
    else:
        part = score.parts[0]

    # Iterate through all elements
    for element in part.flatten().notesAndRests:
        # Skip rests
        if isinstance(element, music21.note.Rest):
            continue

        # Quantize offset and duration to nearest 16th note
        quantized_offset = round(element.offset / sixteenth) * sixteenth
        quantized_duration = round(element.quarterLength / sixteenth) * sixteenth

        # Extract pitch information
        if isinstance(element, music21.note.Note):
            pitch = element.pitch.midi
            notes_list.append({
                "pitch": pitch,
                "duration": quantized_duration,
                "offset": quantized_offset
            })
        elif isinstance(element, music21.chord.Chord):
            pitches = [p.midi for p in element.pitches]
            notes_list.append({
                "pitch": pitches,
                "duration": quantized_duration,
                "offset": quantized_offset
            })

    return notes_list

In [7]:
class Tokenizer:
    """Tokenize music note sequences into integer token IDs."""

    def __init__(self):
        self.vocab = {}
        self.token_to_id = {}
        self.id_to_token = {}
        self.special_tokens = ["<PAD>", "<SOS>", "<EOS>", "<BAR>"]

    def build_vocab(self, all_sequences):
        """
        Build vocabulary from all note sequences.

        Args:
            all_sequences: List of lists of note dicts
        """
        # Initialize with special tokens
        token_id = 0
        for special_token in self.special_tokens:
            self.token_to_id[special_token] = token_id
            self.id_to_token[token_id] = special_token
            token_id += 1

        # Scan all sequences for unique tokens
        tokens = set()
        for sequence in all_sequences:
            for note in sequence:
                pitch = note["pitch"]
                duration = note["duration"]

                if isinstance(pitch, list):
                    # Chord: sorted pitches joined by -
                    sorted_pitches = sorted(pitch)
                    token = f"C_{'_'.join(map(str, sorted_pitches))}_{duration}"
                else:
                    # Single note
                    token = f"N_{pitch}_{duration}"

                tokens.add(token)

        # Add tokens to vocab
        for token in sorted(tokens):
            if token not in self.token_to_id:
                self.token_to_id[token] = token_id
                self.id_to_token[token_id] = token
                token_id += 1

        self.vocab = self.token_to_id.copy()
        print(f"Vocab size: {len(self.vocab)}")

    def encode(self, sequence):
        """
        Convert a list of note dicts to integer token IDs.

        Args:
            sequence: List of note dicts with keys: pitch, duration, offset

        Returns:
            List of integer token IDs
        """
        token_ids = [self.token_to_id["<SOS>"]]

        for note in sequence:
            pitch = note["pitch"]
            duration = note["duration"]

            if isinstance(pitch, list):
                sorted_pitches = sorted(pitch)
                token = f"C_{'_'.join(map(str, sorted_pitches))}_{duration}"
            else:
                token = f"N_{pitch}_{duration}"

            if token in self.token_to_id:
                token_ids.append(self.token_to_id[token])

        token_ids.append(self.token_to_id["<EOS>"])
        return token_ids

    def decode(self, token_ids):
        """
        Convert integer token IDs back to note dicts.

        Args:
            token_ids: List of integer token IDs

        Returns:
            List of note dicts with keys: pitch, duration
        """
        notes = []

        for token_id in token_ids:
            if token_id not in self.id_to_token:
                continue

            token = self.id_to_token[token_id]

            # Skip special tokens except <BAR>
            if token in ["<SOS>", "<EOS>", "<PAD>"]:
                continue

            if token == "<BAR>":
                notes.append({"pitch": None, "duration": 0, "is_bar": True})
            elif token.startswith("N_"):
                # Single note: N_60_0.25
                parts = token[2:].split("_")
                pitch = int(parts[0])
                duration = float(parts[1])
                notes.append({"pitch": pitch, "duration": duration})
            elif token.startswith("C_"):
                # Chord: C_60_64_67_0.5
                parts = token[2:].split("_")
                duration = float(parts[-1])
                pitches = [int(p) for p in parts[:-1]]
                notes.append({"pitch": pitches, "duration": duration})

        return notes

    def save(self, path):
        """Save vocab to JSON file."""
        # Convert id_to_token keys to strings for JSON serialization
        data = {
            "token_to_id": self.token_to_id,
            "id_to_token": {str(k): v for k, v in self.id_to_token.items()}
        }
        with open(path, "w") as f:
            json.dump(data, f)

    def load(self, path):
        """Load vocab from JSON file."""
        with open(path, "r") as f:
            data = json.load(f)

        self.token_to_id = data["token_to_id"]
        self.id_to_token = {int(k): v for k, v in data["id_to_token"].items()}
        self.vocab = self.token_to_id.copy()

In [8]:
class ChordMelodyDataset(torch.utils.data.Dataset):
    """PyTorch Dataset for chord-melody pairs."""

    def __init__(self, pairs, tokenizer, max_seq_len, pad_token_id):
        """
        Args:
            pairs: List of (chord_token_ids, melody_token_ids) tuples
            tokenizer: Tokenizer instance
            max_seq_len: Maximum sequence length
            pad_token_id: Token ID for padding
        """
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.pad_token_id = pad_token_id

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        chord_ids, melody_ids = self.pairs[idx]

        # Truncate or pad chord sequence
        if len(chord_ids) > self.max_seq_len:
            chord_ids = chord_ids[:self.max_seq_len]
        else:
            chord_ids = chord_ids + [self.pad_token_id] * (self.max_seq_len - len(chord_ids))

        # Truncate or pad melody sequence
        if len(melody_ids) > self.max_seq_len:
            melody_ids = melody_ids[:self.max_seq_len]
        else:
            melody_ids = melody_ids + [self.pad_token_id] * (self.max_seq_len - len(melody_ids))

        return {
            "src": torch.tensor(chord_ids, dtype=torch.long),
            "tgt": torch.tensor(melody_ids, dtype=torch.long)
        }

In [9]:
# Full preprocessing pipeline
print("Starting preprocessing pipeline...")

# 1. Find MIDI pairs
pairs = find_midi_pairs(CONFIG["RAW_DIR"])

if len(pairs) == 0:
    print("No MIDI pairs found. Creating dummy data for testing...")
    # Create dummy data for demonstration
    pairs = [
        {"chords": None, "melody": None, "key": "C"},
        {"chords": None, "melody": None, "key": "G"}
    ]

print(f"Processing {len(pairs)} MIDI pairs...")

# Lists to store sequences
all_chord_seqs = []
all_melody_seqs = []
encoded_pairs = []

# 2-3. Process each pair
for i, pair in enumerate(pairs):
    if pair["chords"] is None or pair["melody"] is None:
        continue

    try:
        # Transpose to C
        chord_score = transpose_to_c(pair["chords"], pair["key"])
        melody_score = transpose_to_c(pair["melody"], pair["key"])

        if chord_score is None or melody_score is None:
            continue

        # Extract note sequences
        chord_seq = extract_note_sequence(chord_score, "chords")
        melody_seq = extract_note_sequence(melody_score, "melody")

        all_chord_seqs.append(chord_seq)
        all_melody_seqs.append(melody_seq)

    except Exception as e:
        print(f"Error processing pair {i}: {e}")

print(f"Successfully processed {len(all_chord_seqs)} pairs")

# 4. Build tokenizer
print("Building tokenizer...")
tokenizer = Tokenizer()
all_sequences = all_chord_seqs + all_melody_seqs
tokenizer.build_vocab(all_sequences)

# 5. Encode all sequences
print("Encoding sequences...")
for chord_seq, melody_seq in zip(all_chord_seqs, all_melody_seqs):
    chord_ids = tokenizer.encode(chord_seq)
    melody_ids = tokenizer.encode(melody_seq)
    encoded_pairs.append((chord_ids, melody_ids))

# 6. Create dataset
print("Creating dataset...")
dataset = ChordMelodyDataset(
    encoded_pairs,
    tokenizer,
    CONFIG["MAX_SEQ_LEN"],
    tokenizer.token_to_id["<PAD>"]
)

# 7. Split 80/10/10
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

# 8. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

# Print dataset sizes
print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset)}")
print(f"  Val:   {len(val_dataset)}")
print(f"  Test:  {len(test_dataset)}")
print(f"  Total: {total_size}")

Starting preprocessing pipeline...
Found 5 MIDI pairs
Processing 5 MIDI pairs...
Successfully processed 5 pairs
Building tokenizer...
Vocab size: 32
Encoding sequences...
Creating dataset...

Dataset sizes:
  Train: 4
  Val:   0
  Test:  1
  Total: 5


In [10]:
class ChordToMelodyTransformer(nn.Module):
    """Transformer model for chord-to-melody generation."""

    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, max_seq_len, pad_token_id):
        """
        Args:
            vocab_size: Size of vocabulary
            embed_dim: Embedding dimension
            num_heads: Number of attention heads
            num_layers: Number of transformer layers
            max_seq_len: Maximum sequence length
            pad_token_id: Token ID for padding
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id

        # Embeddings
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = self._create_positional_encoding(max_seq_len, embed_dim)

        # Transformer
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)

        # Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # Output projection
        self.output_proj = nn.Linear(embed_dim, vocab_size)

        self._print_param_count()

    def _create_positional_encoding(self, max_len, d_model):
        """Create sinusoidal positional encoding."""
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # (1, max_len, d_model)

    def _print_param_count(self):
        """Print total parameter count."""
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model parameters: {total_params:,} (trainable: {trainable_params:,})")

    def _get_src_mask(self, src):
        """Create padding mask for source."""
        return (src == self.pad_token_id)

    def _get_tgt_mask(self, tgt):
        """Create causal mask for target (decoder)."""
        batch_size, seq_len = tgt.shape
        # Causal mask: mask future tokens
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=tgt.device, dtype=torch.bool), diagonal=1)
        # Padding mask
        padding_mask = (tgt == self.pad_token_id)
        return causal_mask, padding_mask

    def forward(self, src, tgt):
        """
        Forward pass.

        Args:
            src: Source (chord) token IDs, shape (batch_size, src_seq_len)
            tgt: Target (melody) token IDs, shape (batch_size, tgt_seq_len)

        Returns:
            logits: (batch_size, tgt_seq_len, vocab_size)
        """
        # Embed and add positional encoding
        src_embed = self.embedding(src)
        src_embed = src_embed + self.positional_encoding[:, :src_embed.shape[1], :].to(src.device)

        tgt_embed = self.embedding(tgt)
        tgt_embed = tgt_embed + self.positional_encoding[:, :tgt_embed.shape[1], :].to(tgt.device)

        # Create masks
        src_key_padding_mask = self._get_src_mask(src)
        tgt_causal_mask, tgt_padding_mask = self._get_tgt_mask(tgt)

        # Encode source
        memory = self.encoder(
            src_embed,
            src_key_padding_mask=src_key_padding_mask
        )

        # Decode target
        decoder_out = self.decoder(
            tgt_embed,
            memory,
            tgt_mask=tgt_causal_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        # Project to vocabulary
        logits = self.output_proj(decoder_out)
        return logits

    def generate(self, src, max_len, sos_token_id, eos_token_id, device):
        """
        Greedy decoding: generate melody from chord sequence.

        Args:
            src: Source token IDs, shape (batch_size, src_seq_len) or (src_seq_len,)
            max_len: Maximum generation length
            sos_token_id: Start-of-sequence token ID
            eos_token_id: End-of-sequence token ID
            device: Device to run on

        Returns:
            Generated token IDs, shape (batch_size, generated_len)
        """
        self.eval()

        # Handle single sequence (add batch dim if needed)
        if src.dim() == 1:
            src = src.unsqueeze(0)

        batch_size = src.shape[0]
        src = src.to(device)

        # Encode source once
        src_embed = self.embedding(src)
        src_embed = src_embed + self.positional_encoding[:, :src_embed.shape[1], :].to(device)
        src_key_padding_mask = self._get_src_mask(src)
        memory = self.encoder(src_embed, src_key_padding_mask=src_key_padding_mask)

        # Initialize with <SOS>
        tgt = torch.full((batch_size, 1), sos_token_id, dtype=torch.long, device=device)

        with torch.no_grad():
            for step in range(max_len - 1):
                # Embed target
                tgt_embed = self.embedding(tgt)
                tgt_embed = tgt_embed + self.positional_encoding[:, :tgt_embed.shape[1], :].to(device)

                # Create masks for current target
                tgt_causal_mask, tgt_padding_mask = self._get_tgt_mask(tgt)

                # Decode
                decoder_out = self.decoder(
                    tgt_embed,
                    memory,
                    tgt_mask=tgt_causal_mask,
                    tgt_key_padding_mask=tgt_padding_mask,
                    memory_key_padding_mask=src_key_padding_mask
                )

                # Get next token (greedy)
                logits = self.output_proj(decoder_out[:, -1, :])  # (batch_size, vocab_size)
                next_token = logits.argmax(dim=-1, keepdim=True)  # (batch_size, 1)

                # Append to target
                tgt = torch.cat([tgt, next_token], dim=1)

                # Stop if all sequences generated <EOS>
                if (next_token == eos_token_id).all():
                    break

        return tgt

In [11]:
import matplotlib.pyplot as plt
from pathlib import Path

# Create output directory
Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChordToMelodyTransformer(
    vocab_size=len(tokenizer.vocab),
    embed_dim=CONFIG["EMBED_DIM"],
    num_heads=CONFIG["NUM_HEADS"],
    num_layers=CONFIG["NUM_LAYERS"],
    max_seq_len=CONFIG["MAX_SEQ_LEN"],
    pad_token_id=tokenizer.token_to_id["<PAD>"]
).to(device)

# Loss function (ignore PAD tokens)
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id["<PAD>"])

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=CONFIG["LR"])

# Training loop
train_losses = []
val_losses = []
best_val_loss = float("inf")

print(f"Training on {device}...")
print(f"Total epochs: {CONFIG['EPOCHS']}\n")

for epoch in range(CONFIG["EPOCHS"]):
    # Training phase
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        # Shift tgt: input is all but last token, target is all but first token
        tgt_input = tgt[:, :-1]
        tgt_target = tgt[:, 1:]

        # Forward pass
        logits = model(src, tgt_input)  # (batch_size, seq_len-1, vocab_size)

        # Compute loss
        loss = loss_fn(logits.reshape(-1, len(tokenizer.vocab)), tgt_target.reshape(-1))

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # Validation phase
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            src = batch["src"].to(device)
            tgt = batch["tgt"].to(device)

            tgt_input = tgt[:, :-1]
            tgt_target = tgt[:, 1:]

            logits = model(src, tgt_input)
            loss = loss_fn(logits.reshape(-1, len(tokenizer.vocab)), tgt_target.reshape(-1))

            val_loss += loss.item()

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    # Print progress
    print(f"Epoch {epoch+1}/{CONFIG['EPOCHS']} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model_path = Path(CONFIG["OUTPUT_DIR"]) / "best_model.pt"
        torch.save(model.state_dict(), model_path)
        print(f"  -> Best model saved (val loss: {val_loss:.4f})")

print("\nTraining complete!")

# Plot train vs val loss
plt.figure(figsize=(10, 6))
plt.plot(range(1, CONFIG["EPOCHS"] + 1), train_losses, label="Train Loss", marker="o")
plt.plot(range(1, CONFIG["EPOCHS"] + 1), val_losses, label="Val Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(Path(CONFIG["OUTPUT_DIR"]) / "loss_plot.png", dpi=150)
plt.show()

print(f"Loss plot saved to {CONFIG['OUTPUT_DIR']}/loss_plot.png")

Model parameters: 1,396,768 (trainable: 1,396,768)


/var/folders/ks/xf_cz3yd7cl6pq_zpxxwl_6m0000gn/T/ipykernel_34121/3083845296.py:31: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)


Training on cpu...
Total epochs: 100



ZeroDivisionError: float division by zero

In [ ]:
def generate_remix(chord_midi_path, model, tokenizer, output_path, device):
    """
    Generate a remix by creating a new melody from a chord progression.

    Args:
        chord_midi_path: Path to chord MIDI file
        model: ChordToMelodyTransformer model
        tokenizer: Tokenizer instance
        output_path: Path to save output MIDI
        device: Device to run model on
    """
    try:
        # 1. Load and transpose chord MIDI
        # Parse filename to get key
        filename = Path(chord_midi_path).stem
        parts = filename.split("_")
        if len(parts) >= 5:
            key = parts[4]
        else:
            key = "C"

        chord_score = transpose_to_c(chord_midi_path, key)
        if chord_score is None:
            print(f"Failed to transpose {chord_midi_path}")
            return False

        # 2. Extract and encode chord sequence
        chord_seq = extract_note_sequence(chord_score, "chords")
        chord_ids = tokenizer.encode(chord_seq)

        # 3. Generate melody
        chord_tensor = torch.tensor(chord_ids, dtype=torch.long, device=device).unsqueeze(0)
        sos_id = tokenizer.token_to_id["<SOS>"]
        eos_id = tokenizer.token_to_id["<EOS>"]

        model.eval()
        generated_ids = model.generate(
            chord_tensor,
            max_len=CONFIG["MAX_SEQ_LEN"],
            sos_token_id=sos_id,
            eos_token_id=eos_id,
            device=device
        )
        generated_ids = generated_ids[0].cpu().tolist()  # Remove batch dim

        # 4. Decode token sequence back to note dicts
        melody_notes = tokenizer.decode(generated_ids)

        # 5. Create pretty_midi object
        # Load original chord MIDI
        chord_midi = pretty_midi.PrettyMIDI(chord_midi_path)

        # Create new melody instrument (piano)
        melody_instrument = pretty_midi.Instrument(program=0, name="Generated Melody")

        # Add generated notes to melody instrument
        current_time = 0.0
        for note_dict in melody_notes:
            if note_dict.get("is_bar"):
                continue

            pitch = note_dict.get("pitch")
            duration = note_dict.get("duration", 0.25)

            if pitch is None or isinstance(pitch, list):
                continue

            # Create Note object (velocity 60)
            note = pretty_midi.Note(
                velocity=60,
                pitch=pitch,
                start=current_time,
                end=current_time + duration
            )
            melody_instrument.notes.append(note)
            current_time += duration

        # Add melody track to MIDI
        chord_midi.instruments.append(melody_instrument)

        # 6. Write to output
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        chord_midi.write(output_path)

        return True
    except Exception as e:
        print(f"Error generating remix for {chord_midi_path}: {e}")
        return False


# Generate remixes for test set
print("Generating remixes from test set chord files...")

# Get test set indices
test_indices = test_dataset.indices
test_pairs = [encoded_pairs[i] for i in test_indices]

# Map back to original MIDI files
remix_count = 0
for i, pair in enumerate(test_pairs):
    if i >= len(all_chord_seqs):
        break

    # Get original chord file from pairs list
    for j, original_pair in enumerate(pairs):
        if j == test_indices[i] % len(pairs):
            chord_file = original_pair.get("chords")
            if chord_file and Path(chord_file).exists():
                output_file = Path(CONFIG["OUTPUT_DIR"]) / f"remix_{remix_count:03d}.mid"

                success = generate_remix(
                    chord_file,
                    model,
                    tokenizer,
                    str(output_file),
                    device
                )

                if success:
                    print(f"Generated remix {remix_count}: {output_file}")
                    remix_count += 1
            break

print(f"\nGenerated {remix_count} remixes in {CONFIG['OUTPUT_DIR']}")

In [ ]:
def evaluate_melodies(test_dataset, all_chord_seqs, tokenizer, model, device):
    """
    Evaluate generated melodies against reference melodies on test set.

    Computes:
    1. Chord-tone ratio: % of generated notes that match concurrent chord
    2. Note density ratio: generated notes/bar vs reference notes/bar
    3. Unique pitch ratio: unique pitches / total notes

    Args:
        test_dataset: PyTorch dataset with test indices
        all_chord_seqs: List of chord note sequences
        tokenizer: Tokenizer instance
        model: ChordToMelodyTransformer model
        device: Device to run on

    Returns:
        dict with metrics
    """
    test_indices = test_dataset.indices

    chord_tone_ratios = []
    note_density_ratios = []
    unique_pitch_ratios = []

    print("Evaluating test set melodies...")
    print("=" * 70)

    for idx, test_idx in enumerate(test_indices):
        if test_idx >= len(all_chord_seqs):
            continue

        # Get chord sequence
        chord_seq = all_chord_seqs[test_idx]

        # Generate melody
        chord_ids = tokenizer.encode(chord_seq)
        chord_tensor = torch.tensor(chord_ids, dtype=torch.long, device=device).unsqueeze(0)

        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                chord_tensor,
                max_len=CONFIG["MAX_SEQ_LEN"],
                sos_token_id=tokenizer.token_to_id["<SOS>"],
                eos_token_id=tokenizer.token_to_id["<EOS>"],
                device=device
            )
        generated_ids = generated_ids[0].cpu().tolist()

        # Decode to note dicts
        generated_melody = tokenizer.decode(generated_ids)

        # Get reference melody from dataset
        pair_data = test_dataset.dataset.pairs[test_idx]
        reference_ids = pair_data[1]  # melody token IDs
        reference_melody = tokenizer.decode(reference_ids)

        # === Metric 1: Chord-tone ratio ===
        # Get pitches from chords for matching
        chord_pitches_by_offset = {}
        for chord_note in chord_seq:
            offset = chord_note.get("offset", 0)
            pitch = chord_note.get("pitch")

            if isinstance(pitch, list):
                if offset not in chord_pitches_by_offset:
                    chord_pitches_by_offset[offset] = set()
                chord_pitches_by_offset[offset].update(pitch)
            else:
                if offset not in chord_pitches_by_offset:
                    chord_pitches_by_offset[offset] = set()
                chord_pitches_by_offset[offset].add(pitch)

        chord_tone_matches = 0
        melody_note_count = 0
        for gen_note in generated_melody:
            if gen_note.get("is_bar"):
                continue

            pitch = gen_note.get("pitch")
            offset = gen_note.get("offset", 0)

            if pitch is None or isinstance(pitch, list):
                continue

            melody_note_count += 1

            # Find matching chord offset (closest)
            closest_offset = min(
                chord_pitches_by_offset.keys(),
                key=lambda x: abs(x - offset),
                default=None
            )

            if closest_offset is not None:
                if pitch in chord_pitches_by_offset[closest_offset]:
                    chord_tone_matches += 1

        chord_tone_ratio = chord_tone_matches / melody_note_count if melody_note_count > 0 else 0
        chord_tone_ratios.append(chord_tone_ratio)

        # === Metric 2: Note density ratio ===
        # Notes per bar (assume 4 beats/bar, quarter note = 1.0)
        bar_length = 4.0

        gen_notes_per_bar = 0
        gen_current_bar = 0
        for note in generated_melody:
            if note.get("is_bar"):
                gen_current_bar += 1
                continue
            if note.get("offset", 0) < (gen_current_bar + 1) * bar_length:
                gen_notes_per_bar += 1

        ref_notes_per_bar = 0
        ref_current_bar = 0
        for note in reference_melody:
            if note.get("is_bar"):
                ref_current_bar += 1
                continue
            if note.get("offset", 0) < (ref_current_bar + 1) * bar_length:
                ref_notes_per_bar += 1

        density_ratio = gen_notes_per_bar / ref_notes_per_bar if ref_notes_per_bar > 0 else 1.0
        note_density_ratios.append(density_ratio)

        # === Metric 3: Unique pitch ratio ===
        gen_pitches = []
        for note in generated_melody:
            if note.get("is_bar"):
                continue
            pitch = note.get("pitch")
            if pitch is not None and not isinstance(pitch, list):
                gen_pitches.append(pitch)

        unique_gen_pitches = len(set(gen_pitches))
        total_gen_notes = len(gen_pitches)

        unique_pitch_ratio = unique_gen_pitches / total_gen_notes if total_gen_notes > 0 else 0
        unique_pitch_ratios.append(unique_pitch_ratio)

    # Compute averages
    avg_chord_tone = np.mean(chord_tone_ratios) if chord_tone_ratios else 0
    avg_density = np.mean(note_density_ratios) if note_density_ratios else 0
    avg_unique_pitch = np.mean(unique_pitch_ratios) if unique_pitch_ratios else 0

    # Print results
    print(f"\n{'Metric':<35} {'Score':<15} {'Interpretation'}")
    print("-" * 70)
    print(f"{'Chord-Tone Ratio':<35} {avg_chord_tone:<15.4f} {'(% notes in chord)'}")
    print(f"{'Note Density Ratio':<35} {avg_density:<15.4f} {'(1.0 = same as reference)'}")
    print(f"{'Unique Pitch Ratio':<35} {avg_unique_pitch:<15.4f} {'(variety, 1.0 = max)'}")
    print("=" * 70)

    results = {
        "chord_tone_ratio": avg_chord_tone,
        "note_density_ratio": avg_density,
        "unique_pitch_ratio": avg_unique_pitch,
        "num_test_samples": len(chord_tone_ratios)
    }

    return results


# Run evaluation
print("\nRunning evaluation on test set...")
eval_results = evaluate_melodies(test_dataset, all_chord_seqs, tokenizer, model, device)

print(f"\nEvaluation complete on {eval_results['num_test_samples']} test samples")

SyntaxError: unterminated triple-quoted string literal (detected at line 156) (2039210340.py, line 5)